In [ ]:
# =============================================================================
# CELL 1: SETUP AND CONFIGURATION
# =============================================================================
import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from scipy.stats import pearsonr, PermutationMethod, BootstrapMethod

# --- CONFIGURATION ---
RESULTS_ROOT = r"/workspace/project/shared_data"
OUTDIR = r"./PLOTS_REGRESSION_STRATIFIED_for_paper"

# Choose the threshold to analyze here
SELECTED_THR = "dens0.3"

# External file with diagnosiss / metadata
# Enter here the file containing the Diagnosis_amyloid column
DIAGNOSIS_FILE = r"/workspace/project/shared_data"

# Name of the ID column in the diagnosiss file
# If the file already has the 'id' column, leave it as is.
# Se invece ha per esempio 'SUBJECT_ID', cambia qui.
DIAGNOSIS_ID_COL = "Subject"

# Name of the diagnosiss column in the metadata file
DIAGNOSIS_COL = "Diagnosis_amyloid"

# Group label mapping for plots
DIAGNOSIS_LABEL_MAP = {
    "CN-": "CN Aβ−",
    "CN+": "CN Aβ+",
    "MCI+": "MCI Aβ+",
    "Dementia+": "AD Aβ+"
}

# Mode and score order
MODES_ORDER = ["multilayer", "structural+functional", "structural", "functional"]
SCORES = ["MOCA", "MMSE", "ADAS13", "CDRSB", "PET_CENTILOID"]

# Order of diagnosiss groups in plots
DIAGNOSIS_ORDER = ["CN Aβ−", "CN Aβ+", "MCI Aβ+", "AD Aβ+"]

# Brain plot configuration
ATLAS_NII = r"/workspace/project/shared_data"
NODELABEL_CSV = r"/workspace/project/shared_data"
LABELBASE = "zero"   # "zero" or "one"
Z_THR = 1          # plot only ROI with unified(z) > Z_THR

# --- NILEARN CHECK ---
try:
    import nibabel as nib
    from nilearn import image
    from nilearn.plotting import plot_glass_brain
    HAVE_NILEARN = True
    print("Nilearn available.")
except ImportError:
    HAVE_NILEARN = False
    print("WARNING: Nilearn not found. Brain plots will not be generated.")

# Create output folder
os.makedirs(OUTDIR, exist_ok=True)

print(f"Output directory: {OUTDIR}")
print(f"Selected threshold: {SELECTED_THR}")
print(f"Diagnosis files: {DIAGNOSIS_FILE}")
print(f"Diagnosis ID columns: {DIAGNOSIS_ID_COL}")
print(f"Diagnosis columns: {DIAGNOSIS_COL}")


Nilearn available.
Output directory: ./PLOTS_REGRESSION_STRATIFIED_for_paper
Selected threshold: dens0.3
Diagnosis files: /workspace/project/shared_data
Diagnosis ID columns: Subject
Diagnosis columns: Diagnosis_amyloid


In [2]:
# =============================================================================
# CELL 2: DATA LOADING (Run only once!)
# =============================================================================

def parse_thr(thr_str):
    m = re.match(r"^(dens|thr)\s*([0-9]*\.?[0-9]+)$", str(thr_str).strip())
    if not m:
        return (None, np.nan)
    return (m.group(1), float(m.group(2)))

def load_results_for_thr(results_root, thr_str):
    pred_path = os.path.join(results_root, thr_str, f"predictions_{thr_str}.csv")
    met_path  = os.path.join(results_root, thr_str, f"metrics_{thr_str}.csv")
    coef_path = os.path.join(results_root, thr_str, f"enet_coefs_{thr_str}.csv")

    if not (os.path.exists(pred_path) and os.path.exists(met_path)):
        return None, None, None

    pred = pd.read_csv(pred_path)
    met = pd.read_csv(met_path)
    coefs = pd.read_csv(coef_path) if os.path.exists(coef_path) else None

    # Filter only ElasticNetCV
    if "model" in pred.columns:
        pred = pred[pred["model"] == "ElasticNetCV"].copy()
    if "model" in met.columns:
        met = met[met["model"] == "ElasticNetCV"].copy()
    if coefs is not None and "model" in coefs.columns:
        coefs = coefs[coefs["model"] == "ElasticNetCV"].copy()

    return pred, met, coefs

def load_diagnosiss_table(diagnosiss_file, diagnosiss_id_col="id", diagnosiss_col="Diagnosis_amyloid"):
    if not os.path.exists(diagnosiss_file):
        raise FileNotFoundError(f"Diagnosis files non trovato: {diagnosiss_file}")

    ext = os.path.splitext(diagnosiss_file)[1].lower()

    if ext == ".csv":
        diag = pd.read_csv(diagnosiss_file)
    elif ext in [".xlsx", ".xls"]:
        diag = pd.read_excel(diagnosiss_file, sheet_name="ALL")
    else:
        raise ValueError("Il file diagnosis deve essere .csv, .xlsx oppure .xls")

    if diagnosiss_id_col not in diag.columns:
        raise ValueError(
            f"ID column '{diagnosiss_id_col}' not found in the diagnosis files. "
            f"Available columns: {list(diag.columns)}"
        )

    if diagnosiss_col not in diag.columns:
        raise ValueError(
            f"Diagnosis columns '{diagnosiss_col}' not found in the diagnosis files. "
            f"Available columns: {list(diag.columns)}"
        )

    diag = diag[[diagnosiss_id_col, diagnosiss_col]].copy()
    diag = diag.rename(columns={
        diagnosiss_id_col: "id",
        diagnosiss_col: "Diagnosis_amyloid"
    })

    diag["id"] = diag["id"].astype(str).str.strip()
    diag["Diagnosis_amyloid"] = diag["Diagnosis_amyloid"].astype(str).str.strip()

    diag = diag.drop_duplicates(subset="id")

    return diag

# --- LOADING ONLY THE SELECTED THRESHOLD ---
thr_list = [SELECTED_THR]
print(f"Threshold to load: {thr_list}")

all_metrics_list = []
all_preds_list = []
thr_to_coefs = {}

for thr_str in thr_list:
    print(f"Loading {thr_str}...", end="\r")
    pred, met, coefs = load_results_for_thr(RESULTS_ROOT, thr_str)

    if pred is not None:
        all_preds_list.append(pred)
        all_metrics_list.append(met)
        thr_to_coefs[thr_str] = coefs

if not all_preds_list:
    raise ValueError(f"No data loaded for {SELECTED_THR}. Check the paths.")

# Final concatenation
all_metrics_df = pd.concat(all_metrics_list, ignore_index=True)
all_preds_df = pd.concat(all_preds_list, ignore_index=True)

# Normalize threshold name across possible thr_str conventions
if "thr_str" in all_preds_df.columns and "thr_str" not in all_preds_df.columns:
    all_preds_df = all_preds_df.rename(columns={"thr_str": "thr_str"})
if "thr_str" in all_metrics_df.columns and "thr_str" not in all_metrics_df.columns:
    all_metrics_df = all_metrics_df.rename(columns={"thr_str": "thr_str"})

# Normalize prediction IDs
if "id" not in all_preds_df.columns:
    raise ValueError("La colonna 'id' non è presente nel file predictions.")

all_preds_df["id"] = all_preds_df["id"].astype(str).str.strip()

# --- LOAD DIAGNOSIS FILE AND MERGE ---
diag_df = load_diagnosiss_table(
    DIAGNOSIS_FILE,
    diagnosiss_id_col=DIAGNOSIS_ID_COL,
    diagnosiss_col=DIAGNOSIS_COL
)

all_preds_df = all_preds_df.merge(diag_df, on="id", how="left")

# Diagnosis labels formatted for plots
all_preds_df["Diagnosis_amyloid_label"] = (
    all_preds_df["Diagnosis_amyloid"]
    .map(DIAGNOSIS_LABEL_MAP)
    .fillna(all_preds_df["Diagnosis_amyloid"])
)

# Optional column useful to check which rows have no match
all_preds_df["has_diagnosiss"] = all_preds_df["Diagnosis_amyloid"].notna()

# --- FINAL INFO ---
n_total = len(all_preds_df)
n_with_diag = int(all_preds_df["has_diagnosiss"].sum())
n_without_diag = int((~all_preds_df["has_diagnosiss"]).sum())

print(f"\nDone! Data loaded.")
print(f"Metrics shape: {all_metrics_df.shape}")
print(f"Predictions shape: {all_preds_df.shape}")
print(f"Coefs caricati per: {list(thr_to_coefs.keys())}")
print(f"Predictions con diagnosis associata: {n_with_diag}/{n_total}")
print(f"Predictions senza diagnosis associata: {n_without_diag}/{n_total}")

if "Diagnosis_amyloid_label" in all_preds_df.columns:
    print("\nDiagnosis groups found:")
    print(sorted(all_preds_df["Diagnosis_amyloid_label"].dropna().unique().tolist()))


Threshold to load: ['dens0.3']
Loading dens0.3...
Done! Data loaded.
Metrics shape: (200, 17)
Predictions shape: (5880, 11)
Coefs caricati per: ['dens0.3']
Predictions con diagnosis associata: 5880/5880
Predictions senza diagnosis associata: 0/5880

Diagnosis groups found:
['AD Aβ+', 'CN Aβ+', 'CN Aβ−', 'MCI Aβ+']


In [5]:
# =============================================================================
# CELL 3: SCATTER PLOTS AND METRICS (Statistical Analysis)
# =============================================================================
from scipy.stats import bootstrap, gaussian_kde
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from matplotlib.ticker import FormatStrFormatter

# Only the requested scores
SCORES = ["ADAS13", "CDRSB"]


# Diagnosis group order
DIAGNOSIS_ORDER = ["CN Aβ−", "CN Aβ+", "MCI Aβ−", "MCI Aβ+", "AD Aβ−", "AD Aβ+"]


# Diagnosis group palette
GROUP_COLORS = {
    "CN Aβ−": "#6C9AFF",
    "CN Aβ+": "#54A24B",
    "MCI Aβ−": "#54A24B",
    "MCI Aβ+": "#ECA82C",
    "AD Aβ−": "#B279A2",
    "AD Aβ+": "#E45756"
}


REG_LINE_COLOR = "#222222"
BAND_COLOR = "#9A9A9A"
IDENTITY_COLOR = "#BDBDBD"
MISSING_COLOR = "#9E9E9E"


def compute_regression_band(x, y, xs, alpha=0.05, n_boot=2000, random_state=42):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    n = len(x)
    if n < 3 or np.std(x) < 1e-12:
        return None, None, None

    m, q = np.polyfit(x, y, 1)
    y_fit = m * xs + q

    rng = np.random.default_rng(random_state)
    boot_preds = []

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        xb = x[idx]
        yb = y[idx]

        if np.std(xb) < 1e-12:
            continue

        try:
            mb, qb = np.polyfit(xb, yb, 1)
            boot_preds.append(mb * xs + qb)
        except Exception:
            continue

    if len(boot_preds) < 50:
        return y_fit, None, None

    boot_preds = np.asarray(boot_preds)
    lower = np.percentile(boot_preds, 100 * (alpha / 2), axis=0)
    upper = np.percentile(boot_preds, 100 * (1 - alpha / 2), axis=0)

    return y_fit, lower, upper


def compute_stats(y, yhat):
    mae = mean_absolute_error(y, yhat)
    mse = mean_squared_error(y, yhat)
    rmse = np.sqrt(mse)
    r2 = r2_score(y, yhat) if np.std(y) > 1e-12 else np.nan

    try:
        perm_method = PermutationMethod(n_resamples=10000)
        boot_method = BootstrapMethod(method="BCa")
        res = pearsonr(y, yhat, method=perm_method)
        corr = res.statistic
        p_perm = res.pvalue
        try:
            ci = res.confidence_interval(confidence_level=0.95, method=boot_method)
            ci_low, ci_high = ci.low, ci.high
        except Exception:
            ci_low, ci_high = np.nan, np.nan
    except Exception:
        corr, p_perm, ci_low, ci_high = np.nan, np.nan, np.nan, np.nan

    try:
        def r2_statistic(y_boot, yhat_boot):
            if np.std(y_boot) > 1e-12:
                return r2_score(y_boot, yhat_boot)
            return np.nan

        res_boot_r2 = bootstrap(
            (y, yhat),
            r2_statistic,
            paired=True,
            n_resamples=10000,
            method="BCa",
            confidence_level=0.95
        )
        r2_ci_low = res_boot_r2.confidence_interval.low
        r2_ci_high = res_boot_r2.confidence_interval.high
    except Exception:
        r2_ci_low, r2_ci_high = np.nan, np.nan

    return {
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
        "r2": r2,
        "r2_ci_low": r2_ci_low,
        "r2_ci_high": r2_ci_high,
        "corr": corr,
        "corr_ci_low": ci_low,
        "corr_ci_high": ci_high,
        "p_perm": p_perm
    }


def make_joint_axes(fig, outer_spec):
    gs = outer_spec.subgridspec(
        2, 2,
        height_ratios=[1.15, 4.0],
        width_ratios=[4.0, 1.15],
        hspace=0.05,
        wspace=0.05
    )
    ax_top = fig.add_subplot(gs[0, 0])
    ax_main = fig.add_subplot(gs[1, 0], sharex=ax_top)
    ax_right = fig.add_subplot(gs[1, 1], sharey=ax_main)
    ax_corner = fig.add_subplot(gs[0, 1])
    ax_corner.axis("off")
    return ax_main, ax_top, ax_right


def compute_axis_limits(values, pad_frac=0.04):
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]

    if len(vals) == 0:
        return None, None

    vmin = np.min(vals)
    vmax = np.max(vals)

    if vmax > vmin:
        pad = (vmax - vmin) * pad_frac
    else:
        pad = 1.0

    return vmin - pad, vmax + pad


def _plot_group_kde_top(ax, values, color, x_grid, weight_scale):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return

    if len(values) == 1 or np.std(values) < 1e-12:
        center = values[0]
        width = max((x_grid.max() - x_grid.min()) * 0.01, 1e-6)
        y_curve = np.exp(-0.5 * ((x_grid - center) / width) ** 2)
        y_curve = y_curve / np.trapz(y_curve, x_grid)
    else:
        kde = gaussian_kde(values)
        y_curve = kde(x_grid)

    y_curve = y_curve * weight_scale
    ax.plot(x_grid, y_curve, color=color, linewidth=2.0)
    ax.fill_between(x_grid, 0, y_curve, color=color, alpha=0.12)


def _plot_group_kde_right(ax, values, color, y_grid, weight_scale):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return

    if len(values) == 1 or np.std(values) < 1e-12:
        center = values[0]
        width = max((y_grid.max() - y_grid.min()) * 0.01, 1e-6)
        x_curve = np.exp(-0.5 * ((y_grid - center) / width) ** 2)
        x_curve = x_curve / np.trapz(x_curve, y_grid)
    else:
        kde = gaussian_kde(values)
        x_curve = kde(y_grid)

    x_curve = x_curve * weight_scale
    ax.plot(x_curve, y_grid, color=color, linewidth=2.0)
    ax.fill_betweenx(y_grid, 0, x_curve, color=color, alpha=0.12)


def plot_scatter_by_score(pred_df, outdir, thr_str, score, modes_order):
    sub = pred_df[
        (pred_df["thr_str"] == thr_str) &
        (pred_df["score"] == score)
    ].copy()

    if sub.empty:
        return

    fig = plt.figure(figsize=(16, 14))
    outer = GridSpec(2, 2, figure=fig, hspace=0.22, wspace=0.18)

    legend_groups_present = set()
    missing_present = False

    for panel_idx, mode in enumerate(modes_order):
        row = panel_idx // 2
        col = panel_idx % 2
        ax_main, ax_top, ax_right = make_joint_axes(fig, outer[row, col])

        d = sub[sub["mode"] == mode].copy()
        d = d[np.isfinite(d["y_true"]) & np.isfinite(d["y_pred"])].copy()

        if d.empty:
            ax_main.set_axis_off()
            ax_top.set_axis_off()
            ax_right.set_axis_off()
            continue

        y = d["y_true"].to_numpy(float)
        yhat = d["y_pred"].to_numpy(float)
        stats = compute_stats(y, yhat)

        xmin, xmax = compute_axis_limits(y, pad_frac=0.04)
        ymin, ymax = compute_axis_limits(yhat, pad_frac=0.04)

        x_grid = np.linspace(xmin, xmax, 400)
        y_grid = np.linspace(ymin, ymax, 400)

        n_total = len(d)

        for grp in DIAGNOSIS_ORDER:
            dg = d[d["Diagnosis_amyloid_label"] == grp]
            if dg.empty:
                continue

            legend_groups_present.add(grp)
            color = GROUP_COLORS.get(grp, "#777777")
            weight_scale = len(dg) / n_total

            ax_main.scatter(
                dg["y_true"],
                dg["y_pred"],
                s=30,
                alpha=0.72,
                color=color,
                edgecolor="white",
                linewidth=0.4,
                label=grp
            )

            _plot_group_kde_top(
                ax_top,
                dg["y_true"].to_numpy(float),
                color=color,
                x_grid=x_grid,
                weight_scale=weight_scale
            )

            _plot_group_kde_right(
                ax_right,
                dg["y_pred"].to_numpy(float),
                color=color,
                y_grid=y_grid,
                weight_scale=weight_scale
            )

        d_missing = d[d["Diagnosis_amyloid_label"].isna()]
        if not d_missing.empty:
            missing_present = True
            weight_scale = len(d_missing) / n_total

            ax_main.scatter(
                d_missing["y_true"],
                d_missing["y_pred"],
                s=26,
                alpha=0.50,
                color=MISSING_COLOR,
                edgecolor="white",
                linewidth=0.3,
                label="Missing diagnosiss"
            )

            _plot_group_kde_top(
                ax_top,
                d_missing["y_true"].to_numpy(float),
                color=MISSING_COLOR,
                x_grid=x_grid,
                weight_scale=weight_scale
            )

            _plot_group_kde_right(
                ax_right,
                d_missing["y_pred"].to_numpy(float),
                color=MISSING_COLOR,
                y_grid=y_grid,
                weight_scale=weight_scale
            )

        x_id_min = min(xmin, ymin)
        x_id_max = max(xmax, ymax)
        ax_main.plot(
            [x_id_min, x_id_max],
            [x_id_min, x_id_max],
            linestyle="--",
            linewidth=1.2,
            color=IDENTITY_COLOR,
            zorder=0
        )

        if len(y) >= 3 and np.std(y) > 1e-12:
            xs = np.linspace(xmin, xmax, 300)
            y_fit, lower, upper = compute_regression_band(y, yhat, xs)

            if lower is not None and upper is not None:
                ax_main.fill_between(
                    xs, lower, upper,
                    color=BAND_COLOR,
                    alpha=0.10,
                    zorder=1
                )
                ax_main.plot(
                    xs, lower,
                    color=BAND_COLOR,
                    linewidth=0.9,
                    linestyle="--",
                    alpha=0.65,
                    zorder=2
                )
                ax_main.plot(
                    xs, upper,
                    color=BAND_COLOR,
                    linewidth=0.9,
                    linestyle="--",
                    alpha=0.65,
                    zorder=2
                )

            if y_fit is not None:
                ax_main.plot(
                    xs, y_fit,
                    color=REG_LINE_COLOR,
                    linewidth=2.4,
                    alpha=0.95,
                    zorder=3
                )

        if np.isnan(stats["p_perm"]):
            ptxt = "p=NA"
        elif stats["p_perm"] < 1e-20:
            ptxt = "p<1e-20"
        else:
            ptxt = f'p={stats["p_perm"]:.3e}'

        title_str = (
            f"{mode}\n"
            f'MAE={stats["mae"]:.3f} | RMSE={stats["rmse"]:.3f}\n'
            f'R²={stats["r2"]:.3f} [{stats["r2_ci_low"]:.3f}, {stats["r2_ci_high"]:.3f}]\n'
            f'r={stats["corr"]:.3f} [{stats["corr_ci_low"]:.3f}, {stats["corr_ci_high"]:.3f}] | {ptxt}'
        )
        #ax_top.set_title(title_str, fontsize=10)

        ax_main.set_xlim(xmin, xmax)
        ax_main.set_ylim(ymin, ymax)
        ax_top.set_xlim(xmin, xmax)
        ax_right.set_ylim(ymin, ymax)

        ax_main.grid(alpha=0.20)
        ax_main.set_xlabel("True", fontsize=16)
        ax_main.set_ylabel("Predicted", fontsize=16)
        ax_main.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))
        ax_top.set_ylabel("Dens.", fontsize=16)
        ax_right.set_xlabel("Dens.", fontsize=16)
        ax_main.tick_params(axis='both', labelsize=12)
        ax_top.tick_params(axis='both', labelsize=12)
        ax_right.tick_params(axis='both', labelsize=12)
        plt.setp(ax_top.get_xticklabels(), visible=False)
        plt.setp(ax_right.get_yticklabels(), visible=False)

        ax_top.spines["right"].set_visible(False)
        ax_top.spines["top"].set_visible(False)
        ax_right.spines["right"].set_visible(False)
        ax_right.spines["top"].set_visible(False)
        ax_main.spines["right"].set_visible(False)
        ax_main.spines["top"].set_visible(False)

        ax_top.grid(alpha=0.10)
        ax_right.grid(alpha=0.10)

    legend_handles = [
        Line2D([0], [0], marker='o', color='w',
               markerfacecolor=GROUP_COLORS[g], markeredgecolor='white',
               markeredgewidth=0.5, markersize=8, label=g)
        for g in DIAGNOSIS_ORDER if g in legend_groups_present
    ]

    if missing_present:
        legend_handles.append(
            Line2D([0], [0], marker='o', color='w',
                   markerfacecolor=MISSING_COLOR, markeredgecolor='white',
                   markeredgewidth=0.5, markersize=8, label="Missing diagnosiss")
        )

    if legend_handles:
        fig.legend(
            handles=legend_handles,
            loc="upper center",
            ncol=min(4, len(legend_handles)),
            frameon=False,
            bbox_to_anchor=(0.5, 0.98),
            fontsize=14
        )

    fig.suptitle(f"{score} | {thr_str} | ElasticNetCV (Pooled Folds)", fontsize=16, y=0.995)
    fig.tight_layout(rect=[0, 0, 1, 0.965])

    os.makedirs(outdir, exist_ok=True)
    fig.savefig(os.path.join(outdir, f"SCATTER_{thr_str}_{score}.png"), dpi=300, bbox_inches="tight")
    plt.close(fig)


def plot_metrics_vs_thr(all_metrics_df, outdir, score, metric_name, modes_order):
    df = all_metrics_df[
        (all_metrics_df["model"] == "ElasticNetCV") &
        (all_metrics_df["score"] == score)
    ].copy()

    if df.empty or metric_name not in df.columns:
        return

    g = (
        df.groupby(["thr_str", "mode"], as_index=False)[metric_name]
        .mean()
        .rename(columns={metric_name: f"{metric_name}_mean"})
    )

    g[["thr_type", "thr_val"]] = g["thr_str"].apply(lambda s: pd.Series(parse_thr(s)))

    for thr_type in ["dens", "thr"]:
        gg = g[g["thr_type"] == thr_type].copy()
        if gg.empty:
            continue

        fig, axes = plt.subplots(2, 2, figsize=(10, 9), sharex=True)
        axes = axes.ravel()

        for ax, mode in zip(axes, modes_order):
            dm = gg[gg["mode"] == mode].sort_values("thr_val")
            if dm.empty:
                ax.set_axis_off()
                continue

            ax.plot(dm["thr_val"], dm[f"{metric_name}_mean"], marker="o")
            ax.set_title(mode)
            ax.set_xlabel(thr_type)
            ax.set_ylabel(f"mean {metric_name}")
            ax.grid(alpha=0.25)

        fig.suptitle(f"ElasticNetCV | {score} | {metric_name} vs {thr_type}")
        #fig.tight_layout(rect=[0, 0, 1, 0.95])

        os.makedirs(outdir, exist_ok=True)
        fig.savefig(
            os.path.join(outdir, f"METRICS_{score}_{metric_name}_vs_{thr_type}.png"),
            dpi=300,
            bbox_inches="tight"
        )
        plt.close(fig)


# --- ESECUZIONE PLOTS ---
print("Generating scatter plots...")
out_scatter = os.path.join(OUTDIR, "scatter")

unique_thrs = sorted(all_preds_df["thr_str"].dropna().unique())

for thr in unique_thrs:
    for score in SCORES:
        plot_scatter_by_score(all_preds_df, out_scatter, thr, score, MODES_ORDER)

print(f"Scatter plots saved in: {out_scatter}")


# --- METRICS VS THRESHOLD ---
print("Generating metric-vs-threshold plots...")
out_met = os.path.join(OUTDIR, "metrics_vs_thr")

for score in SCORES:
    for metric_name in ["mae_test", "rmse_test", "r2_test", "corr_test"]:
        if metric_name in all_metrics_df.columns:
            plot_metrics_vs_thr(all_metrics_df, out_met, score, metric_name, MODES_ORDER)

print(f"Metric plots saved in: {out_met}")


Generating scatter plots...


C:\workspace\project
  fig.tight_layout(rect=[0, 0, 1, 0.965])
C:\workspace\project
  fig.tight_layout(rect=[0, 0, 1, 0.965])


Scatter plots saved in: ./PLOTS_REGRESSION_STRATIFIED_for_paper\scatter
Generating metric-vs-threshold plots...
Metric plots saved in: ./PLOTS_REGRESSION_STRATIFIED_for_paper\metrics_vs_thr


In [3]:
# =============================================================================
# CELL 4: BRAIN PLOTS & ROI EXPORT
# =============================================================================

import json
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

try:
    import nibabel as nib
    from nilearn import image
    from nilearn.plotting import plot_stat_map, plot_glass_brain
    HAVE_NILEARN = True
except Exception:
    HAVE_NILEARN = False


# -----------------------------------------------------------------------------
# Color configuration: same as classification
# -----------------------------------------------------------------------------
COLOR_FUNC = "#3b82f6"
COLOR_STRUCT = "#ef4444"
COLOR_BOTH = "#10b981"
COLOR_ML_ONLY = "#a855f7"

BRAIN_DPI = 400
TRANSPARENT_BRAIN = "glass"
GLASS_DISPLAY_MODE = "lzry"
GLASS_PLOT_ABS = False


# -----------------------------------------------------------------------------
# Regex and helper functions
# -----------------------------------------------------------------------------
RE_NODE = re.compile(r"(?:^|_)(\d+)$")


def _extract_node(feat: str):
    m = RE_NODE.search(str(feat))
    if not m:
        nums = re.findall(r"\d+", str(feat))
        if not nums:
            return None
        try:
            return int(nums[-1])
        except Exception:
            return None
    try:
        return int(m.group(1))
    except Exception:
        return None


def _split_sf(feat: str):
    node = _extract_node(feat)
    if node is None:
        return None, None, None
    base = RE_NODE.sub("", str(feat))
    if base.startswith("func_"):
        return "func", base[len("func_"):], node
    if base.startswith("struct_"):
        return "struct", base[len("struct_"):], node
    return None, base, node


def _normalize(values: dict, how: str = "zscore") -> dict:
    if how == "none":
        return dict(values)

    arr = np.array(list(values.values()), dtype=float)
    if arr.size == 0:
        return {}

    if how == "zscore":
        mu = float(np.mean(arr))
        sd = float(np.std(arr))
        if sd == 0:
            return {n: 0.0 for n in values}
        return {n: (v - mu) / sd for n, v in values.items()}

    if how == "minmax":
        mn = float(np.min(arr))
        mx = float(np.max(arr))
        if mx == mn:
            return {n: 0.0 for n in values}
        return {n: (v - mn) / (mx - mn) for n, v in values.items()}

    raise ValueError(f"Unsupported normalization: {how}")


def _fuse_metrics(metric_to_values: dict, normalization: str = "zscore") -> dict:
    if not metric_to_values:
        return {}
    normed = {m: _normalize(d, normalization) for m, d in metric_to_values.items()}
    nodes = set().union(*[set(d.keys()) for d in normed.values()]) if normed else set()

    fused = {}
    for n in nodes:
        vals = [d[n] for d in normed.values() if n in d]
        if vals:
            fused[n] = float(np.mean(vals))
    return fused


def _find_feature_col(df):
    for c in ["feature", "feat", "name", "connection", "predictor"]:
        if c in df.columns:
            return c
    raise ValueError("No feature column found in the coefficients dataframe.")


def _find_value_col(df):
    for c in ["coef", "weight", "beta", "importance", "coef_mean", "value"]:
        if c in df.columns:
            return c
    raise ValueError("No coefficient/importance column found in the dataframe.")


def _aggregate_by_mode_regression(df: pd.DataFrame, mode: str, score: str) -> dict:
    sub = df.copy()

    if "mode" in sub.columns:
        sub = sub[sub["mode"] == mode].copy()
    if "score" in sub.columns:
        sub = sub[sub["score"] == score].copy()
    if "model" in sub.columns:
        sub = sub[sub["model"] == "ElasticNetCV"].copy()

    if sub.empty:
        return {}

    feat_col = _find_feature_col(sub)
    val_col = _find_value_col(sub)

    agg = sub.groupby(feat_col)[val_col].mean().reset_index(name="imp")
    grouped = {}

    if mode == "structural+functional":
        for _, r in agg.iterrows():
            feat = r[feat_col]
            val = float(r["imp"])
            side, base, node = _split_sf(feat)
            if node is None:
                continue
            if side not in {"func", "struct"}:
                side = "struct"
            metric_dict = grouped.setdefault(base, {"func": {}, "struct": {}})
            metric_dict[side][node] = max(abs(val), metric_dict[side].get(node, -np.inf))
    else:
        for _, r in agg.iterrows():
            feat = r[feat_col]
            val = float(r["imp"])
            node = _extract_node(feat)
            if node is None:
                continue
            base = RE_NODE.sub("", str(feat))
            d = grouped.setdefault(base, {})
            d[node] = max(abs(val), d.get(node, -np.inf))

    return grouped


def _unified_from_sf(grouped_sf: dict, normalization: str = "zscore"):
    func_metrics, struct_metrics = {}, {}
    for metric, sides in grouped_sf.items():
        if sides.get("func"):
            func_metrics[metric] = sides["func"]
        if sides.get("struct"):
            struct_metrics[metric] = sides["struct"]
    return _fuse_metrics(func_metrics, normalization), _fuse_metrics(struct_metrics, normalization)


def load_mapping(mapping_csv: str, label_base: str, max_node: int):
    if mapping_csv is None or not os.path.exists(mapping_csv):
        start = 1 if label_base == "one" else 0
        node_to_label = {i: i for i in range(start, max_node + 1)}
        node_to_name = {i: None for i in range(start, max_node + 1)}
        return node_to_label, node_to_name

    m = pd.read_csv(mapping_csv)

    if not {"node", "atlas_label"} <= set(m.columns):
        raise ValueError("NODELABEL_CSV must contain columns: node, atlas_label")

    node_to_label = dict(zip(m["node"].astype(int), m["atlas_label"].astype(int)))
    if "label" in m.columns:
        node_to_name = dict(zip(m["node"].astype(int), m["label"].astype(str)))
    else:
        node_to_name = {int(n): None for n in m["node"].astype(int)}

    return node_to_label, node_to_name


def build_roi_img(atlas_img, node_to_label: dict, values_per_node: dict):
    data = atlas_img.get_fdata()
    out = np.zeros_like(data, dtype=float)

    for node, val in values_per_node.items():
        lab = node_to_label.get(int(node), None)
        if lab is None:
            continue
        out[data == lab] = float(val)

    return image.new_img_like(atlas_img, out)


def _mono_cmap(hex_color: str):
    import matplotlib.colors as mcolors
    base = mcolors.to_rgb(hex_color)
    return ListedColormap([(1, 1, 1, 0), base])


def _plot_panel_glass(ax, img, cmap, vmax):
    if img is None:
        ax.axis("off")
        return

    plot_glass_brain(
        img,
        display_mode=GLASS_DISPLAY_MODE,
        colorbar=False,
        plot_abs=GLASS_PLOT_ABS,
        vmax=vmax,
        cmap=cmap,
        alpha=1,
        axes=ax,
    )


def _plot_panel_stat(ax, img, cmap, vmin, vmax):
    if img is None:
        ax.axis("off")
        return

    plot_stat_map(
        img,
        display_mode="ortho",
        cut_coords=4,
        axes=ax,
        colorbar=False,
        annotate=False,
        draw_cross=False,
        bg_img=None,
        black_bg=(TRANSPARENT_BRAIN == "overlay"),
        dim=0,
        vmin=vmin,
        vmax=vmax,
        cmap=cmap,
    )


def _plot_four_panel_regression(atlas_img, node_to_label, sets_dict, title, out_png):
    struct_nodes = sets_dict["structural_only"]
    func_nodes = sets_dict["functional_only"]
    sf_struct = sets_dict["sf_struct"]
    sf_func = sets_dict["sf_func"]
    sf_both = sets_dict["sf_both"]
    ml_f = sets_dict["ml_f"]
    ml_s = sets_dict["ml_s"]
    ml_fs = sets_dict["ml_fs"]
    ml_none = sets_dict["ml_none"]

    if HAVE_NILEARN and atlas_img is not None and node_to_label is not None:
        struct_img = build_roi_img(atlas_img, node_to_label, {n: 1.0 for n in struct_nodes}) if struct_nodes else None
        func_img = build_roi_img(atlas_img, node_to_label, {n: 1.0 for n in func_nodes}) if func_nodes else None

        sf_values = {}
        for n in sf_func:
            sf_values[n] = 1.0
        for n in sf_struct:
            sf_values[n] = 2.0
        for n in sf_both:
            sf_values[n] = 3.0
        sf_img = build_roi_img(atlas_img, node_to_label, sf_values) if sf_values else None

        ml_values = {}
        for n in ml_f:
            ml_values[n] = 1.0
        for n in ml_s:
            ml_values[n] = 2.0
        for n in ml_fs:
            ml_values[n] = 3.0
        for n in ml_none:
            ml_values[n] = 4.0
        ml_img = build_roi_img(atlas_img, node_to_label, ml_values) if ml_values else None

        cmap_tri = ListedColormap([COLOR_FUNC, COLOR_STRUCT, COLOR_BOTH])
        cmap_quad = ListedColormap([COLOR_FUNC, COLOR_STRUCT, COLOR_BOTH, COLOR_ML_ONLY])

        if TRANSPARENT_BRAIN == "glass":
            fig = plt.figure(figsize=(8.5, 8.5))
            gs = fig.add_gridspec(
                5, 1,
                height_ratios=[1, 1, 1, 1, 0.16],
                left=0.02, right=0.99, top=0.98, bottom=0.02,
                hspace=0.03
            )
            ax1 = fig.add_subplot(gs[0, 0])
            ax2 = fig.add_subplot(gs[1, 0])
            ax3 = fig.add_subplot(gs[2, 0])
            ax4 = fig.add_subplot(gs[3, 0])
            axL = fig.add_subplot(gs[4, 0])
            axL.axis("off")

            _plot_panel_glass(ax1, struct_img, _mono_cmap(COLOR_STRUCT), 1.0)
            _plot_panel_glass(ax2, func_img, _mono_cmap(COLOR_FUNC), 1.0)
            _plot_panel_glass(ax3, sf_img, cmap_tri, 3.5)
            _plot_panel_glass(ax4, ml_img, cmap_quad, 4.5)

            #ax1.set_title("structural", fontsize=11)
            #ax2.set_title("functional", fontsize=11)
            #ax3.set_title("structural+functional", fontsize=11)
            #ax4.set_title("multilayer", fontsize=11)

            legend_handles = [
                Patch(facecolor=COLOR_STRUCT, label="Structural"),
                Patch(facecolor=COLOR_FUNC, label="Functional"),
                Patch(facecolor=COLOR_BOTH, label="Both"),
                Patch(facecolor=COLOR_ML_ONLY, label="ML-only"),
            ]
            axL.legend(handles=legend_handles, loc="center", frameon=False, ncol=4)

           # fig.suptitle(title, fontsize=14, y=0.995)
            fig.savefig(out_png, dpi=BRAIN_DPI, bbox_inches="tight", pad_inches=0.01)
            plt.close(fig)
            return [out_png]

        else:
            fig = plt.figure(figsize=(10, 10))
            gs = fig.add_gridspec(
                5, 1,
                height_ratios=[1, 1, 1, 1, 0.16],
                left=0.03, right=0.97, top=0.97, bottom=0.03,
                hspace=0.08
            )
            ax1 = fig.add_subplot(gs[0, 0])
            ax2 = fig.add_subplot(gs[1, 0])
            ax3 = fig.add_subplot(gs[2, 0])
            ax4 = fig.add_subplot(gs[3, 0])
            axL = fig.add_subplot(gs[4, 0])
            axL.axis("off")

            _plot_panel_stat(ax1, struct_img, _mono_cmap(COLOR_STRUCT), 0.0, 1.0)
            _plot_panel_stat(ax2, func_img, _mono_cmap(COLOR_FUNC), 0.0, 1.0)
            _plot_panel_stat(ax3, sf_img, cmap_tri, 0.5, 3.5)
            _plot_panel_stat(ax4, ml_img, cmap_quad, 0.5, 4.5)

            #ax1.set_title("structural", fontsize=11)
            #ax2.set_title("functional", fontsize=11)
            #ax3.set_title("structural+functional", fontsize=11)
            #ax4.set_title("multilayer", fontsize=11)

            legend_handles = [
                Patch(facecolor=COLOR_STRUCT, label="Structural"),
                Patch(facecolor=COLOR_FUNC, label="Functional"),
                Patch(facecolor=COLOR_BOTH, label="Both"),
                Patch(facecolor=COLOR_ML_ONLY, label="ML-only"),
            ]
            axL.legend(handles=legend_handles, loc="center", frameon=False, ncol=4)

            fig.suptitle(title, fontsize=14, y=0.995)
            fig.savefig(out_png, dpi=BRAIN_DPI, bbox_inches="tight", pad_inches=0.01)
            plt.close(fig)
            return [out_png]

    fig = plt.figure(figsize=(10, 8))
    gs = fig.add_gridspec(2, 2, wspace=0.25, hspace=0.35)
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[1, 0])
    ax4 = fig.add_subplot(gs[1, 1])

    def _bar(ax, nodes, color, ttl):
        if nodes:
            xs = sorted(nodes)
            ax.bar(xs, [1] * len(xs), color=color)
            ax.set_title(ttl)
            ax.set_xlabel("node")
            ax.set_yticks([])
        else:
            ax.set_title(ttl + " (none)")
            ax.axis("off")

    _bar(ax1, struct_nodes, COLOR_STRUCT, "structural")
    _bar(ax2, func_nodes, COLOR_FUNC, "functional")

    for lab, color, nodes in [
        ("functional", COLOR_FUNC, sf_func),
        ("structural", COLOR_STRUCT, sf_struct),
        ("both", COLOR_BOTH, sf_both)
    ]:
        if nodes:
            ax3.bar(sorted(nodes), [1] * len(nodes), color=color, label=lab)
    ax3.legend(frameon=False)
    #ax3.set_title("structural+functional")
    ax3.set_xlabel("node")
    ax3.set_yticks([])

    for lab, color, nodes in [
        ("F", COLOR_FUNC, ml_f),
        ("S", COLOR_STRUCT, ml_s),
        ("FS", COLOR_BOTH, ml_fs),
        ("ML-only", COLOR_ML_ONLY, ml_none)
    ]:
        if nodes:
            ax4.bar(sorted(nodes), [1] * len(nodes), color=color, label=lab)
    ax4.legend(frameon=False)
    #ax4.set_title("multilayer")
    ax4.set_xlabel("node")
    ax4.set_yticks([])

    fig.suptitle(title, fontsize=14)
    fig.savefig(out_png, dpi=BRAIN_DPI, bbox_inches="tight")
    plt.close(fig)
    return [out_png]


def _export_regression_roi_table(thr_str, score, outdir, node_to_name, z_thr, modalities):
    rows = []
    all_nodes = set()
    for _, d in modalities.items():
        all_nodes |= set(d.keys())

    for node in sorted(all_nodes):
        row = {
            "thr_str": thr_str,
            "score": score,
            "node": int(node),
            "label": node_to_name.get(int(node)) if node_to_name else None,
            "z_threshold": float(z_thr),
        }
        for name, d in modalities.items():
            val = d.get(node, np.nan)
            row[f"importance_{name}"] = val
            row[f"selected_{name}"] = bool((not pd.isna(val)) and (val >= z_thr))
        rows.append(row)

    out_csv = os.path.join(outdir, f"ROI_importance_{thr_str}_{score}.csv")
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    return out_csv


# -----------------------------------------------------------------------------
# Main execution
# -----------------------------------------------------------------------------
if HAVE_NILEARN and os.path.exists(ATLAS_NII):
    print("Inizio generazione Brain Plots...")
    atlas_img = nib.load(ATLAS_NII)

    all_nodes = []
    for thr_str, coefs_df in thr_to_coefs.items():
        if coefs_df is None or coefs_df.empty:
            continue
        feat_col = None
        for c in ["feature", "feat", "name", "connection", "predictor"]:
            if c in coefs_df.columns:
                feat_col = c
                break
        if feat_col is not None:
            all_nodes.extend([n for n in coefs_df[feat_col].dropna().apply(_extract_node).tolist() if n is not None])

    node_to_label, node_to_name = load_mapping(
        NODELABEL_CSV if os.path.exists(NODELABEL_CSV) else None,
        LABELBASE,
        max(all_nodes) if all_nodes else 0
    )

    all_roi_rows = []
    out_brain = os.path.join(OUTDIR, "brain")
    os.makedirs(out_brain, exist_ok=True)

    saved_all = []
    csv_all = []

    for thr_str in sorted(thr_to_coefs.keys()):
        coefs_df = thr_to_coefs[thr_str]
        if coefs_df is None or coefs_df.empty:
            continue

        print(f"Processing Brains for {thr_str}...", end="\r")

        for score in SCORES:
            grouped_struct = _aggregate_by_mode_regression(coefs_df, "structural", score)
            grouped_func = _aggregate_by_mode_regression(coefs_df, "functional", score)
            grouped_sf = _aggregate_by_mode_regression(coefs_df, "structural+functional", score)
            grouped_ml = _aggregate_by_mode_regression(coefs_df, "multilayer", score)

            struct_u = _fuse_metrics(grouped_struct, "zscore") if grouped_struct else {}
            func_u = _fuse_metrics(grouped_func, "zscore") if grouped_func else {}
            sf_func_u, sf_struct_u = _unified_from_sf(grouped_sf, "zscore") if grouped_sf else ({}, {})
            ml_u = _fuse_metrics(grouped_ml, "zscore") if grouped_ml else {}

            csv_path = _export_regression_roi_table(
                thr_str=thr_str,
                score=score,
                outdir=out_brain,
                node_to_name=node_to_name,
                z_thr=Z_THR,
                modalities={
                    "structural": struct_u,
                    "functional": func_u,
                    "sf_structural": sf_struct_u,
                    "sf_functional": sf_func_u,
                    "multilayer": ml_u,
                },
            )
            csv_all.append(csv_path)

            structural_only = {n for n, s in struct_u.items() if s >= Z_THR}
            functional_only = {n for n, s in func_u.items() if s >= Z_THR}
            sf_struct_sel = {n for n, s in sf_struct_u.items() if s >= Z_THR}
            sf_func_sel = {n for n, s in sf_func_u.items() if s >= Z_THR}
            ml_sel = {n for n, s in ml_u.items() if s >= Z_THR}

            sf_both = sf_struct_sel & sf_func_sel
            sf_struct = sf_struct_sel - sf_both
            sf_func = sf_func_sel - sf_both

            ml_fs = ml_sel & sf_struct_sel & sf_func_sel
            ml_f = (ml_sel & sf_func_sel) - sf_struct_sel
            ml_s = (ml_sel & sf_struct_sel) - sf_func_sel
            ml_none = ml_sel - (ml_fs | ml_f | ml_s)

            sets_dict = {
                "structural_only": structural_only,
                "functional_only": functional_only,
                "sf_struct": sf_struct,
                "sf_func": sf_func,
                "sf_both": sf_both,
                "ml_f": ml_f,
                "ml_s": ml_s,
                "ml_fs": ml_fs,
                "ml_none": ml_none,
            }

            out_png = os.path.join(out_brain, f"OVERLAP_REGRESSION_{thr_str}_{score}.png")
            saved = _plot_four_panel_regression(
                atlas_img=atlas_img,
                node_to_label=node_to_label,
                sets_dict=sets_dict,
                title=f"{score} | {thr_str} | ElasticNetCV",
                out_png=out_png
            )
            saved_all.extend(saved)

            for lab, nodes in sets_dict.items():
                for n in sorted(nodes):
                    all_roi_rows.append({
                        "thr_str": thr_str,
                        "score": score,
                        "set": lab,
                        "node": int(n),
                        "label": node_to_name.get(int(n)) if node_to_name else None,
                    })

    if all_roi_rows:
        df_roi = pd.DataFrame(all_roi_rows)
        df_roi.to_excel(os.path.join(out_brain, f"ALL_ROI_Zthr{Z_THR}.xlsx"), index=False)
        print("\nBrain analysis completata.")
    else:
        print("\nNessuna ROI estratta.")

    manifest = {
        "figures": saved_all,
        "csv_files": csv_all,
        "outdir": out_brain
    }
    with open(os.path.join(out_brain, "manifest_regression_brain.json"), "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

else:
    print("Brain plots saltati (Nilearn mancante o Atlas non trovato).")


Inizio generazione Brain Plots...
Processing Brains for dens0.3...
Brain analysis completata.
